In [9]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

In [ ]:
from datasets import load_dataset

dataset = load_dataset('Blpeng/nsmc')
for i in range(3):
    print(dataset['train'][i]['document'], dataset['train'][i]['label'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


아 더빙.. 진짜 짜증나네요 목소리 0
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나 1
너무재밓었다그래서보는것을추천한다 0


In [2]:
from transformers import AutoTokenizer
MODEL_NAME = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_col = 'document'
format_columns = ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
def tokenize_fn(batch):
    """배치 단위로 텍스트 토크나이즈 (최대 128 토큰)"""
    # None/비문자열 샘플이 섞여 있어도 안전하게 문자열로 변환
    texts = [x if isinstance(x, str) else '' for x in batch[text_col]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=128,
        padding='max_length'
    )
# 전체 데이터셋에 적용 (batched=True로 빠르게)
tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')  # Trainer가 'labels' 필드 기대
tokenized.set_format('torch', columns=format_columns)

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/150000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(f'모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'분류 헤드: {model.classifier}')

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 파라미터 수: 110,618,882
분류 헤드: Linear(in_features=768, out_features=2, bias=True)


In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./nsmc_bert',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch', 
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    report_to='none'
)


In [7]:
import numpy as np
from transformers import Trainer
import evaluate

accuracy_metric = evaluate.load('accuracy')
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 속도를 위해 1000개 샘플만 사용
small_train = tokenized['train'].select(range(5000))
small_eval  = tokenized['test'].select(range(5000))
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics
)

In [8]:
train_result = trainer.train()

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)